In [ ]:
import torch

data_stored = "bbq-l31-65k.pt"
data = torch.load(f"../activations/{data_stored}", weights_only=False)

# SAE activations are stored as sparse tensors — convert to dense
sae_activations = [act.to_dense() for act in data["sae_activations"]]
generations     = data["generations"]
categories      = data["categories"]
model_config    = data["model_config"]
sae_config      = data["sae_config"]

print(f"Loaded {len(sae_activations)} samples")
print(f"Model: {model_config['model_name']}")
print(f"SAE:   layer {sae_config['layer']}, width {sae_config['width']}, L0 {sae_config['l0']}")

In [ ]:
CATEGORY = "Physical_appearance"

# Indices belonging to the target category
cat_indices = [i for i, c in enumerate(categories) if c == CATEGORY]

# Slice every data list to keep only those samples
sae_activations = [sae_activations[i] for i in cat_indices]
generations     = [generations[i]     for i in cat_indices]
categories      = [categories[i]      for i in cat_indices]

print(f"Kept {len(cat_indices)} samples for '{CATEGORY}'")
print(f"First generation preview:\n{generations[0][:300]}")


In [ ]:
from tqdm import tqdm
from src.aggregator import Aggregator
from src.denoiser import Denoiser
from src.configs import SAEConfig
from src.neuronpedia_client import NeuronpediaClient, build_sae_id
from src.feature import Feature

aggregator = Aggregator()
aggregated = torch.stack([
    aggregator.max(act)
    for act in tqdm(sae_activations, desc="Aggregating")
])

print(f"Aggregated matrix shape: {aggregated.shape}")

# Build Neuronpedia client and denoiser
model_id = model_config["model_name"].split("/")[-1]
sae_cfg = SAEConfig(
    repo_id=sae_config["repo_id"],
    sae_type=sae_config["sae_type"],
    layer=sae_config["layer"],
    width=sae_config["width"],
    l0=sae_config["l0"],
)
client = NeuronpediaClient(model_id=model_id, sae_id=build_sae_id(sae_cfg))
denoiser = Denoiser(neuronpedia_client=client)

PROMPT_IDX = 1
TOP_K = 6000

#normalised = denoiser.tf_idf(aggregated)
top_strengths, top_indices = aggregated[PROMPT_IDX].topk(TOP_K)
features = Feature.from_activations(top_indices, top_strengths, client)

print(f"Prompt #{PROMPT_IDX}  |  category: {categories[PROMPT_IDX]}")
print(f"\nGeneration:\n{generations[PROMPT_IDX]}\n")
print(f"Top {TOP_K} SAE features:")
for f in features:
    desc = f.description or "(no description)"
    print(f"  Feature {f.feature_idx:>6d}  score={f.strength:+.3f}  ->  {desc} -> url {f.url}")